In [9]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, f1_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [4]:
path_db = '../db/filmes.db'

engine = create_engine(f'sqlite:///{path_db}')

In [5]:
df = pd.read_sql('SELECT * from filmes', con=engine)
df

,id,title,original_title,original_language,overview,budget,revenue,runtime,release_date,genres,popularity,vote_average,vote_count,num_genres,release_year
0,411405,Small Crimes,Small Crimes,en,"A disgraced former cop, fresh off a six-year p...",8.201666e+06,4.243626e+06,95.0,2017-04-28 00:00:00.000000,"['Drama', 'Comedy', 'Thriller', 'Crime']",7.219022,5.8,55.0,4,2017.0
1,42492,Up the Sandbox,Up the Sandbox,en,"A young wife and mother, bored with day-to-day...",3.543621e+05,2.978317e+06,97.0,1972-12-21 00:00:00.000000,"['Drama', 'Comedy']",0.138450,7.3,2.0,2,1972.0
2,12143,Bad Lieutenant,Bad Lieutenant,en,"While investigating a young nun's rape, a corr...",1.000000e+06,2.019469e+06,96.0,1992-09-16 00:00:00.000000,"['Crime', 'Drama']",6.417037,6.9,162.0,2,1992.0
3,9976,Satan's Little Helper,Satan's Little Helper,en,A naïve young boy unknowingly becomes the pawn...,1.890390e+06,1.710526e+06,100.0,2004-01-01 00:00:00.000000,"['Horror', 'Romance', 'Comedy']",2.233189,5.0,42.0,3,2004.0
4,46761,Sitcom,Sitcom,fr,The adventures of an upper-class suburban fami...,8.550376e+05,2.725346e+06,80.0,1998-05-27 00:00:00.000000,"['Comedy', 'Drama', 'Thriller']",1.800582,6.4,27.0,3,1998.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4993,3093,Basic Instinct 2,Basic Instinct 2,en,Novelist Catherine Tramell is once again in tr...,7.000000e+07,3.862948e+07,114.0,2006-03-29 00:00:00.000000,"['Crime', 'Mystery', 'Thriller']",15.321902,4.6,183.0,3,2006.0
4994,246127,Every Thing Will Be Fine,Every Thing Will Be Fine,en,"One day, driving aimlessly around the outskirt...",9.853274e+06,8.034000e+03,118.0,2015-04-02 00:00:00.000000,['Drama'],5.723103,5.3,79.0,1,2015.0
4995,9803,Seven Dwarfs,7 Zwerge - Männer allein im Wald,de,The Seven Dwarves live deep within a female-fr...,9.674091e+06,1.694475e+06,95.0,2004-10-28 00:00:00.000000,['Comedy'],4.582736,5.2,70.0,1,2004.0
4996,336970,True Siblings,Syskonsalt,sv,"The siblings Linus, 19-years-old, who are taki...",1.060106e+06,4.436564e+06,58.0,2000-09-13 00:00:00.000000,"['Drama', 'TV Movie']",2.364355,8.0,2.0,2,2000.0


In [6]:
df['sucesso'] = df.apply(lambda x: 1 if x['revenue'] > x['budget'] else 0, axis=1)
df

,id,title,original_title,original_language,overview,budget,revenue,runtime,release_date,genres,popularity,vote_average,vote_count,num_genres,release_year,sucesso
0,411405,Small Crimes,Small Crimes,en,"A disgraced former cop, fresh off a six-year p...",8.201666e+06,4.243626e+06,95.0,2017-04-28 00:00:00.000000,"['Drama', 'Comedy', 'Thriller', 'Crime']",7.219022,5.8,55.0,4,2017.0,0
1,42492,Up the Sandbox,Up the Sandbox,en,"A young wife and mother, bored with day-to-day...",3.543621e+05,2.978317e+06,97.0,1972-12-21 00:00:00.000000,"['Drama', 'Comedy']",0.138450,7.3,2.0,2,1972.0,1
2,12143,Bad Lieutenant,Bad Lieutenant,en,"While investigating a young nun's rape, a corr...",1.000000e+06,2.019469e+06,96.0,1992-09-16 00:00:00.000000,"['Crime', 'Drama']",6.417037,6.9,162.0,2,1992.0,1
3,9976,Satan's Little Helper,Satan's Little Helper,en,A naïve young boy unknowingly becomes the pawn...,1.890390e+06,1.710526e+06,100.0,2004-01-01 00:00:00.000000,"['Horror', 'Romance', 'Comedy']",2.233189,5.0,42.0,3,2004.0,0
4,46761,Sitcom,Sitcom,fr,The adventures of an upper-class suburban fami...,8.550376e+05,2.725346e+06,80.0,1998-05-27 00:00:00.000000,"['Comedy', 'Drama', 'Thriller']",1.800582,6.4,27.0,3,1998.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4993,3093,Basic Instinct 2,Basic Instinct 2,en,Novelist Catherine Tramell is once again in tr...,7.000000e+07,3.862948e+07,114.0,2006-03-29 00:00:00.000000,"['Crime', 'Mystery', 'Thriller']",15.321902,4.6,183.0,3,2006.0,0
4994,246127,Every Thing Will Be Fine,Every Thing Will Be Fine,en,"One day, driving aimlessly around the outskirt...",9.853274e+06,8.034000e+03,118.0,2015-04-02 00:00:00.000000,['Drama'],5.723103,5.3,79.0,1,2015.0,0
4995,9803,Seven Dwarfs,7 Zwerge - Männer allein im Wald,de,The Seven Dwarves live deep within a female-fr...,9.674091e+06,1.694475e+06,95.0,2004-10-28 00:00:00.000000,['Comedy'],4.582736,5.2,70.0,1,2004.0,0
4996,336970,True Siblings,Syskonsalt,sv,"The siblings Linus, 19-years-old, who are taki...",1.060106e+06,4.436564e+06,58.0,2000-09-13 00:00:00.000000,"['Drama', 'TV Movie']",2.364355,8.0,2.0,2,2000.0,1


In [7]:
df.columns

Index(['id', 'title', 'original_title', 'original_language', 'overview',
       'budget', 'revenue', 'runtime', 'release_date', 'genres', 'popularity',
       'vote_average', 'vote_count', 'num_genres', 'release_year', 'sucesso'],
      dtype='object')

In [8]:
features = ['budget', 'revenue', 'runtime', 'popularity', 'vote_average', 'vote_count', 'num_genres', 'release_year']

In [14]:
def modelo_bayes(df, features, coluna_alvo):
    
    valido = ~df[features].isna().any(axis=1) & (df[coluna_alvo].isin({0, 1}))
    
    X = df.loc[valido, features]
    y = df.loc[valido, coluna_alvo]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=42, stratify=y)
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    pipeline = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)), ('model', GaussianNB())])
    
    param_grid = {
        'model__var_smoothing': np.logspace(-9, 1, 9)
    }
    
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring='f1',
        cv=skf,
        n_jobs=-1,
        verbose=2
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f'Melhores parâmetros: {grid_search.best_params_}')
    print(f'Melhor f1_score k-fold (médio): {grid_search.best_score_}')
    
    best_model = grid_search.best_estimator_
    best_model.fit(X_train, y_train)
    
    y_pred = best_model.predict(X_test)
    
    print(classification_report(y_test, y_pred))
    
    return {'gs': grid_search,
            'bm': best_model}

In [15]:
bayes_gauss = modelo_bayes(df, features, 'sucesso')
bayes_gauss

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Melhores parâmetros: {'model__var_smoothing': 0.03162277660168379}
Melhor f1_score k-fold (médio): 0.3251332349271345
              precision    recall  f1-score   support

           0       0.30      0.93      0.45       259
           1       0.89      0.21      0.34       718

    accuracy                           0.40       977
   macro avg       0.59      0.57      0.39       977
weighted avg       0.73      0.40      0.37       977



{'gs': GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
              estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                        ('smote', SMOTE(random_state=42)),
                                        ('model', GaussianNB())]),
              n_jobs=-1,
              param_grid={'model__var_smoothing': array([1.00000000e-09, 1.77827941e-08, 3.16227766e-07, 5.62341325e-06,
        1.00000000e-04, 1.77827941e-03, 3.16227766e-02, 5.62341325e-01,
        1.00000000e+01])},
              scoring='f1', verbose=2),
 'bm': Pipeline(steps=[('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)),
                 ('model', GaussianNB(var_smoothing=0.03162277660168379))])}

Podemos ver que o __f1_score__ do treino final nos dados de teste foi maior do que a estimativa do __gride search__ no treinamento quando foi buscar os melhores hiperparâmetros para o modelo `0.40 > 0.325`. Isso indica que o medelo não está em overfitting, uma vez que a inferência em novos dados teve melhor desempenho do que a estimativa da capacidade de generalização do modelo (treinamento). Por outro lado, o modelo no teste obtendo `f1 score` de `0.40` ainda é considerado baixo e pode indicar que o modelo está tendo dificuldades de separar as classes. 

Em comparação com outros modelos, até o momento este foi o que teve desempenho mais baixo. 

Agora vamos fazer o cálculo manual de uma probabilidade condicional e o dado pelo modelo e compara-las.

In [49]:
x_prova = np.array(x_prova, dtype=float).reshape(1, -1)
print(bayes_gauss['bm'].named_steps['model'].predict_proba(x_prova))

[[0. 1.]]


In [50]:
np.array(x_prova).shape

(1, 8)

In [44]:
means = bayes_gauss['bm'].named_steps['model'].theta_
vars = bayes_gauss['bm'].named_steps['model'].var_
priors = bayes_gauss['bm'].named_steps['model'].class_prior_
classes = bayes_gauss['bm'].named_steps['model'].classes_

print(f'Classes: {classes}')
print(f'Medias por classe: {means}')
print(f'Variância: {vars}')
print(f'Priors: {priors}')

Classes: [0 1]
Medias por classe: [[ 0.21417988 -0.03212262  0.08478747  0.1233507  -0.45559359 -0.07579884
   0.1591494   0.33962265]
 [-0.07794204  0.01158554 -0.0341489  -0.04535954  0.16741574  0.02535858
  -0.05264245 -0.11761546]]
Variância: [[0.76459094 0.03629303 0.33305733 0.17431487 1.34939306 0.10806664
  1.00592098 0.51223905]
 [1.09304609 1.39547165 1.27799952 1.33489782 0.81442173 1.36157823
  1.03319391 1.16205121]]
Priors: [0.5 0.5]


DEfinindo a Gaussian, temos:

In [51]:
def Gaussian_manual(x, mean, var):
    
    x = np.array(x, dtype=float)
    mean = np.array(mean, dtype=float)
    var = np.array(var, dtype=float)
    
    coeff = 1.0 / np.sqrt(2*np.pi*var)
    exp = np.exp(-((x - mean) ** 2) / (2 * var))
    return coeff * exp

In [ ]:
def posterior_prob_manual(x_obs, means, vars, priors, classes):
    
    x_obs = np.array(x_obs, dtype=float)

    log_likelihoods = []
    for c in range(len(classes)):
        
        pdfs = Gaussian_manual(x_obs, means[c], vars[c]) + 1e-300
        log_p_x_given_c = np.sum(np.log(pdfs))
        
        log_posterior_unnorm = log_p_x_given_c + np.log(priors[c])
        log_likelihoods.append(log_posterior_unnorm)

    log_likelihoods = np.array(log_likelihoods)

    max_log = np.max(log_likelihoods)
    log_prob_norm = log_likelihoods - (max_log + np.log(np.sum(np.exp(log_likelihoods - max_log))))
    posterior_probs = np.exp(log_prob_norm)

    return posterior_probs

In [ ]:
posterior_probs = posterior_prob_manual(x_prova, means, vars, priors, classes)

for c, prob in zip(classes, posterior_probs):
    print(f"P(C={c} | x_obs) = {prob:.6f}")

P(C=0 | x_obs) = 0.000000
P(C=1 | x_obs) = 1.000000


Temos que tanto no cálculo manual quanto no cálculo dado pelo scikit_learn o resultado foi o mesmo de `x_prova` ter probabilidade de quase 100% de pertencer a classe `1`. 